# 05_119_챗봇_연동확인
화면 오른쪽 아래 챗봇 창이 쓰는 `/api/chat` 동작 확인

- 먼저 별도 CMD에서 Flask 서버 실행 (`chatbot_exaone.py`가 연결된 상태)
- 이 노트북은 모델을 다시 불러오지 않고, 서버에 질문만 보냄

In [1]:
import sys
import requests
import pandas as pd
from IPython.display import display

# chatbot_exaone.py가 있는 폴더를 파이썬 검색 경로에 추가
CHATBOT_DIR = r"C:\ai\source\09_Project\cheolhyeon_jo\html"
sys.path.insert(0, CHATBOT_DIR)
from chatbot_exaone import find_district, find_topics

BASE_URL = "http://127.0.0.1:5000"

## 1. 질문 분석 확인
질문 속 자치구 이름과 질문 종류(병상 / 이송 / 인구)를 찾는 규칙

In [2]:
questions = ["강남 응급실 병상 알려줘", "마포구 병원 어디 있어?",
             "다음 달 이송 건수 예측은?", "송파구 생활인구 밀집", "안녕하세요"]

display(pd.DataFrame({"질문": questions,
                      "자치구": [find_district(q) for q in questions],
                      "질문 종류": [find_topics(q) for q in questions]}))

,질문,자치구,질문 종류
0,강남 응급실 병상 알려줘,강남구,[beds]
1,마포구 병원 어디 있어?,마포구,[beds]
2,다음 달 이송 건수 예측은?,서울 전체,[transport]
3,송파구 생활인구 밀집,송파구,[population]
4,안녕하세요,서울 전체,"[beds, transport]"


## 2. 모델 상태 확인
ready = EXAONE 답변 / loading = 불러오는 중 / error = 데이터 요약 답변으로 동작

In [3]:
try:
    res = requests.get(BASE_URL + "/api/chat/status", timeout=10)
except requests.exceptions.ConnectionError:
    raise SystemExit(f"{BASE_URL} 서버가 꺼져 있습니다. CMD에서 Flask 서버를 먼저 실행하세요.")

if res.headers.get("Content-Type", "").startswith("application/json"):
    display(pd.DataFrame([res.json()]))
else:
    # json이 아니면 챗봇 주소가 서버에 연결되지 않은 상태
    display(pd.DataFrame([{"응답 코드": res.status_code,
                           "응답 형식": res.headers.get("Content-Type"),
                           "응답 앞부분": res.text[:150]}]))

,응답 코드,응답 형식,응답 앞부분
0,404,text/html; charset=utf-8,<!doctype html>\n<html lang=en>\n<title>404 No...


## 3. 챗봇 질문 테스트
챗봇 창과 같은 방식으로 질문 + 선택 자치구를 보냄 (CPU에서는 한 질문에 수십 초 걸릴 수 있음)

In [6]:
rows = []
for q in questions[:3]:
    res = requests.post(BASE_URL + "/api/chat",
                        json={"message": q, "district": "서울 전체"}, timeout=180)
    if res.headers.get("Content-Type", "").startswith("application/json"):
        data = res.json()
        rows.append({"질문": q, "응답 코드": res.status_code, "자치구": data.get("district"),
                     "답변 방식": data.get("source"), "답변": data.get("answer")})
    else:
        rows.append({"질문": q, "응답 코드": res.status_code, "자치구": None,
                     "답변 방식": "json 아님", "답변": res.text[:150]})

display(pd.DataFrame(rows).style.set_properties(subset=["답변"], **{"white-space": "pre-wrap"}))

,질문,응답 코드,자치구,답변 방식,답변
0,강남 응급실 병상 알려줘,200,강남구,summary,[강남구] 실시간 응급실 병상 (기준 2026-10-02T14:43:14) - 연세대학교의과대학강남세브란스병원 : 7개 - 삼성서울병원 : 0개
1,마포구 병원 어디 있어?,200,마포구,summary,[마포구] 실시간 응급실 병상 (기준 2026-10-02T14:43:14) - 신촌연세병원 : 확인 불가
2,다음 달 이송 건수 예측은?,200,서울 전체,summary,"[서울 전체] 월별 이송 건수 예측 (Ensemble(GB+Ridge)) - 2026-01 : 23,880건 - 2026-02 : 21,642건 - 2026-03 : 22,570건 - 2026-04 : 23,046건 - 2026-05 : 25,136건 - 2026-06 : 24,978건"
